In [4]:
# librerias

!pip install pandas openpyxl -q
!pip install plotly -q
%pip install kaleido -q


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
from openai import OpenAI

client = OpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=api_key_gpt,
)

In [6]:

from openai import OpenAI
import pandas as pd 
import plotly.express as px
import plotly.graph_objects as go
from tqdm import tqdm
import os
import warnings


warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [3]:
import plotly.io as pio

# ✅ Definir template personalizado
pio.templates["mi_estilo"] = go.layout.Template(
    layout=go.Layout(
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(color="white", family="Arial"),
        title=dict(
            font=dict(color="white"),
            x=0,           # ✅ Siempre a la izquierda
            xanchor="left"
        ),
        xaxis=dict(
            title_font=dict(color="white"),
            tickfont=dict(color="white"),
            gridcolor="rgba(255,255,255,0.2)",
            linecolor="rgba(255,255,255,0.3)",
        ),
        yaxis=dict(
            title_font=dict(color="white"),
            tickfont=dict(color="white"),
            linecolor="rgba(255,255,255,0.3)",
        ),
        legend=dict(
            font=dict(color="white"),
            title=dict(font=dict(color="white")),
        ),
    )
)

# ✅ Aplicarlo como default para TODAS las figuras
pio.templates.default = "mi_estilo"

In [7]:
# lectura de información - surtigas

df = pd.read_excel("data/Reporte Histórico de Roturas - Surtigas.xlsx", sheet_name="Detalle Roturas Historico")

In [4]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Fecha de Creación,9350,2024-07-18 22:16:00.203529,2023-01-02 12:26:25,2023-09-19 06:53:37,2024-06-13 13:51:33,2025-05-29 15:36:12.750000,2026-04-08 15:27:13,NaN
Orden de trabajo,9350.0,245092040.085348,111111111.0,220493306.75,241730719.5,270121730.5,999999999.0,31028544.923244
LAT,750.0,9.5473,3.45068,9.190532,9.417365,10.372067,10.78852,0.880472
LONG,750.0,-75.364037,-76.52004,-75.514817,-75.40786,-75.202933,-73.94478,0.35174
LAT2,311.0,9.478309,3.45068,8.94026,9.37583,10.36951,10.78852,0.916865
LONG3,311.0,-75.378397,-76.52004,-75.52418,-75.41307,-75.193955,-73.94478,0.366374


In [5]:
df["year"] = df["Fecha de Creación"].dt.year

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9350 entries, 0 to 9349
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Recurso                    9350 non-null   str           
 1   Fecha de Creación          9350 non-null   datetime64[us]
 2   Distrito                   9350 non-null   str           
 3   Ciudad                     9350 non-null   str           
 4   Diametro                   9350 non-null   object        
 5   Dimensión Rotura           8043 non-null   object        
 6   Orden de trabajo           9350 non-null   int64         
 7   Dirección                  9342 non-null   str           
 8   Sector Operativo           9304 non-null   str           
 9   Hora de asignación         8390 non-null   object        
 10  Hora de Inicio             8271 non-null   object        
 11  Hora Control               8081 non-null   object        
 12  Hora Normalizació

## Analisis Surtigas Ruptura de Tuberia 

In [7]:
colores = ["#481066", "#ea1e76", "#4a7628", "#cfde00", "#00b4e3", "#05205a"]

In [36]:
df_grouped = df.groupby(["year", "Distrito"]).size().reset_index(name="Conteo")
df_grouped["year"] = df_grouped["year"].astype(str)

orden = ["BOLIVAR", "SUCRE", "CORDOBA"]
df_grouped["Distrito"] = pd.Categorical(df_grouped["Distrito"], categories=orden, ordered=True)

# ✅ Calcular el total por Distrito para sacar el porcentaje
totales = df_grouped.groupby("Distrito")["Conteo"].transform("sum")
df_grouped["Porcentaje"] = (df_grouped["Conteo"] / totales * 100).round(1).astype(str) + "%"

fig = px.bar(
    df_grouped,
    x="Conteo",
    y="Distrito",
    color="year",
    orientation="h",
    text="Conteo",           # ✅ Muestra el porcentaje en cada sección
    category_orders={"Distrito": orden},
    color_discrete_sequence=colores,
    title="Rupturas por Distrito y Año"
)

fig.update_traces(
    textposition="inside",           # ✅ Texto dentro de cada barra
    insidetextanchor="middle",       # ✅ Centrado en la sección
    textfont=dict(color="white", size=18, family="Arial"),
)

fig.show()

# ✅ Exportar con fondo transparente (requiere: pip install kaleido)
fig.write_image("plot1.png", scale=2, width=1200, height=400)

/tmp/ipykernel_16529/536642717.py:32: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image("plot1.png", scale=2, width=1200, height=400)


In [42]:
df_grouped = (df.groupby(["Distrito", "Ciudad"]).size().reset_index(name="Conteo"))
df_top5 = (df_grouped.sort_values(["Distrito", "Conteo"], ascending=[True, False]).groupby("Distrito").head(5))

totales = df_top5.groupby("Distrito")["Conteo"].transform("sum")
df_top5["PorcentajeNum"] = (df_top5["Conteo"] / totales * 100).round(1)
df_top5["Porcentaje"] = (df_top5["Conteo"] / totales * 100).round(1).astype(str) + "%"

orden = ["BOLIVAR","SUCRE","CORDOBA"]

df_top5["Distrito"] = pd.Categorical(df_top5["Distrito"], categories=orden, ordered=True)

df_top5["Etiqueta"] = df_top5.apply(
    lambda row: f"{row['Ciudad']}<br>{row['Porcentaje']}" if row["PorcentajeNum"] >= 15 else row["Porcentaje"],
    axis=1
)

fig = px.bar(
    df_top5,
    x="Conteo",
    y="Distrito",
    color="Ciudad",
    orientation="h",
    text="Etiqueta",
    category_orders={"Distrito": orden},
    color_discrete_sequence=colores,
    title="Rupturas por Distrito y Ciudad"
)

fig.update_traces(
    textposition="inside",           # ✅ Texto dentro de cada barra
    insidetextanchor="middle",       # ✅ Centrado en la sección
    textfont=dict(color="white", size=15, family="Arial"),
)

fig.update_layout(
    showlegend=False)

fig.show()

fig.write_image("plot2.png", scale=2, width=1200, height=400)

/tmp/ipykernel_16529/1363067303.py:40: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image("plot2.png", scale=2, width=1200, height=400)


## Analisis causales de rupturas

In [39]:
df["TERCERO/OBRA PUBLICA"] = (
    df["TERCERO/OBRA PUBLICA"]
    .astype(str)              # asegurar string
    .str.strip()              # quitar espacios
    .str.upper()              # estandarizar mayúsculas
)
df_modalidad = (df.groupby(["TERCERO/OBRA PUBLICA"]).size().reset_index(name="Conteo"))

In [53]:
# Gráfico
fig = px.bar(
    df_modalidad,
    x="TERCERO/OBRA PUBLICA",
    y="Conteo",
    text="Conteo",
    color_discrete_sequence=colores,
    title="Rupturas por tipo de intervención"
)



fig.update_traces(
    textposition="outside",
    marker_line_width=0  
)

fig.update_layout(  
    showlegend=False,          
    height=400,
    width=700,
    bargap=0.5,                
    title_x=0.5,
    yaxis=dict(
    gridcolor="rgba(200,200,200,0.2)",
    griddash="dot"       # opciones: "dot", "dash", "longdash", "dashdot"
),                
)

# Ajuste eje Y para dar aire a etiquetas
fig.update_yaxes(range=[0, df_modalidad["Conteo"].max() * 1.2])

fig.show()

fig.write_image("plot3.png", scale=2, width=600, height=400)

/tmp/ipykernel_16529/3608633389.py:35: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image("plot3.png", scale=2, width=600, height=400)


In [8]:
df_tipo = (df.groupby(["TIPO"]).size().sort_values(ascending=False).reset_index(name="Conteo"))

In [9]:
df_tipo

,TIPO,Conteo
0,Excavación,3230
1,Pavimentación,1826
2,Instalación de postes,1631
3,Alcantarillado,1235
4,Acueducto,1008
...,...,...
62,TERCERO,1
63,Vándalo o desadaptado,1
64,instalación de cerramiento,1
65,no identificada,1


In [10]:
categorias = df_tipo["TIPO"].tolist()

In [13]:
prompt = f"""Tengo estas {len(categorias)} categorías de tipos de obra/intervención en infraestructura urbana:

{json.dumps(categorias, ensure_ascii=False, indent=2)}

Agrúpalas en EXACTAMENTE 10 categorías limpias y representativas.
Considera sinónimos, errores ortográficos, variaciones de formato y contexto.

Responde SOLO con un JSON con este formato exacto:
{{
  "categorias_finales": ["Cat1", "Cat2", ..., "Cat10"],
  "mapeo": {{
    "categoria_original": "categoria_final",
    ...
  }}
}}"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    response_format={"type": "json_object"},
    temperature=0,
)

resultado = json.loads(response.choices[0].message.content)

print("✅ 10 Categorías finales:")
for i, cat in enumerate(resultado["categorias_finales"], 1):
    print(f"  {i}. {cat}")

print("\n📋 Mapeo completo:")
for original, final in resultado["mapeo"].items():
    print(f"  '{original}' → '{final}'")

✅ 10 Categorías finales:
  1. Excavación
  2. Pavimentación
  3. Instalaciones
  4. Alcantarillado y Acueducto
  5. Limpieza
  6. Remodelación
  7. Obras Civiles
  8. Daños
  9. Prevención y Mantenimiento
  10. Otros

📋 Mapeo completo:
  'Excavación' → 'Excavación'
  'Pavimentación' → 'Pavimentación'
  'Instalación de postes' → 'Instalaciones'
  'Alcantarillado' → 'Alcantarillado y Acueducto'
  'Acueducto' → 'Alcantarillado y Acueducto'
  'Obras Civiles' → 'Obras Civiles'
  'Otros' → 'Otros'
  'Remodelación vivienda' → 'Remodelación'
  'Instalación de Cerramiento' → 'Instalaciones'
  'Limpieza Caños y canales' → 'Limpieza'
  'No Identificada' → 'Otros'
  'NO APLICA' → 'Otros'
  'Vandalismo' → 'Daños'
  'Causa Natural' → 'Daños'
  'Instalación de ductos eléctricos' → 'Instalaciones'
  'Daño Vehicular' → 'Daños'
  'Instalación de cerramiento' → 'Instalaciones'
  'Limpieza de Maleza' → 'Limpieza'
  'Prevencion de daños' → 'Prevención y Mantenimiento'
  'Incendio' → 'Daños'
  'Limpieza de 

In [90]:
for original, final in resultado["mapeo"].items():
    if final == "Obras civiles y construcción":
        print(original)

Obras Civiles
Obra Civil
obras civiles
Construcción de una bomba
Construcción de Anden
Remodelación vivienda
Remodelacion de vivienda
Remodelacion


In [60]:
# Aplica el mapeo al DataFrame
df["TIPO_LIMPIO"] = df["TIPO"].map(resultado["mapeo"])

In [15]:
import json

with open("mapeo_categorias.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=2)

In [72]:
df_tipo = (df.groupby(["TIPO_LIMPIO"]).size().reset_index(name="Conteo").sort_values("Conteo", ascending=False))

# Gráfico
fig = px.bar(
    df_tipo,
    x="TIPO_LIMPIO",
    y="Conteo",
    text="Conteo",
    color_discrete_sequence=colores,
    title="Rupturas por tipo de intervención"
)


fig.update_traces(
    textposition="outside",
    marker_line_width=0  
)

fig.update_layout(   
    showlegend=False,          
    height=500,
    width=700,
    bargap=0.5,                
    title_x=0.5,
    title = None,
    xaxis_title=None,
    xaxis=dict(tickangle=-35),
    margin=dict(l=60, r=60, t=60, b=200),
    yaxis=dict(
    gridcolor="rgba(200,200,200,0.2)",
    griddash="dot"       # opciones: "dot", "dash", "longdash", "dashdot"
)                 
)

# Ajuste eje Y para dar aire a etiquetas
fig.update_yaxes(range=[0, df_tipo["Conteo"].max() * 1.2])

fig.write_image("plot4.png", scale=2, width=600, height=600)

fig.show()

/tmp/ipykernel_16529/2788063224.py:38: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image("plot4.png", scale=2, width=600, height=600)


In [74]:
from fractions import Fraction

def diametro_a_numero(valor):
    valor = str(valor).strip()
    
    try:
        # caso: "1 1/2"
        if " " in valor:
            entero, fraccion = valor.split()
            return float(entero) + float(Fraction(fraccion))
        
        # caso: "1/2"
        if "/" in valor:
            return float(Fraction(valor))
        
        # caso: número normal "2", "3"
        return float(valor)
    
    except:
        return None

In [79]:
df["Diametro"] = df["Diametro"].astype(str)
df["Diametro"] = df["Diametro"].replace("2026-04-03 00:00:00", "3/4")
df_diametro = (df.groupby(["Diametro"]).size().sort_values(ascending=False).reset_index(name="Conteo"))
df_diametro["Diametro_num"] = df_diametro["Diametro"].apply(diametro_a_numero)
df_diametro = df_diametro.sort_values(by="Diametro_num").drop(columns="Diametro_num")
df_diametro

,Diametro,Conteo
1,1/2,970
0,3/4,7827
5,1,52
8,1 1/4,1
7,1 1/2,3
2,2,301
3,3,122
4,4,68
6,6,6


In [84]:
fig = px.pie(
    df_diametro.sort_values("Conteo", ascending=False),
    names="Diametro",
    values="Conteo",
    color_discrete_sequence=colores,
    title="Rupturas por diámetro",
    hole=0.4,          # ✅ Tamaño del hueco: 0 = pie, 1 = sin relleno. Prueba entre 0.3 y 0.5
)

fig.update_traces(
    textposition="inside",
    textinfo="percent+label",   # ✅ Opciones: "percent", "label", "value", "percent+label"
    textfont=dict(color="white", size=11, family="Arial"),
)

fig.update_layout(
    showlegend=False,
    height=600,
    width=700,
)

fig.write_image("plot5.png", scale=2, width=800, height=600)
fig.show()

/tmp/ipykernel_16529/2350273376.py:22: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image("plot5.png", scale=2, width=800, height=600)


In [85]:
df_sankey = (
    df.groupby(["TERCERO/OBRA PUBLICA", "TIPO_LIMPIO", "Diametro"])
      .size()
      .reset_index(name="Conteo")
)

labels = list(pd.concat([
    df_sankey["TERCERO/OBRA PUBLICA"],
    df_sankey["TIPO_LIMPIO"],
    df_sankey["Diametro"]
]).unique())

label_dict = {label: i for i, label in enumerate(labels)}

## Nivel 1: TERCERO/OBRA PUBLICA → TIPO_LIMPIO
source_1 = df_sankey["TERCERO/OBRA PUBLICA"].map(label_dict)
target_1 = df_sankey["TIPO_LIMPIO"].map(label_dict)
value_1  = df_sankey["Conteo"]

## Nivel 2: TIPO_LIMPIO → Diametro
source_2 = df_sankey["TIPO_LIMPIO"].map(label_dict)
target_2 = df_sankey["Diametro"].map(label_dict)
value_2  = df_sankey["Conteo"]

## RESUMEN:
source = pd.concat([source_1, source_2])
target = pd.concat([target_1, target_2])
value  = pd.concat([value_1, value_2])

In [46]:

# ── Paleta de colores ─────────────────────────────────────────
COLORES_NODO = {
    # Nivel 1 - Responsable
    "OBRA":                                     "#2E91E5",
    "TERCERO":                                  "#E15F99",
    # Nivel 2 - Tipo
    "Pavimentación":                            "#1CA71C",
    "Alcantarillado y Acueducto":               "#00A08B",
    "Instalación de postes y cerramientos":     "#DA16FF",
    "Excavación":                               "#FB8C00",
    "Obras Civiles y construcción":             "#B68100",
    "Remodelación":                             "#750D86",
    "Limpieza y mantenimiento":                 "#EB663B",
    "Daños y vandalismo":                       "#FB0D0D",
    "Otros y no identificados":                 "#90A4AE",
    "Instalación de tuberías y ductos eléctricos":"#222A2A",
    "3/4":                                      "#094799",
}
COLOR_DEFAULT = "#CFD8DC"

# ── Colores nodos ─────────────────────────────────────────────
node_colors = [COLORES_NODO.get(l, COLOR_DEFAULT) for l in labels]

# ── Colores links: hereda color del nodo origen con opacidad ──
def hex_to_rgba(hex_color, alpha=0.25):
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    return f"rgba({r},{g},{b},{alpha})"

link_colors = [hex_to_rgba(node_colors[s]) for s in source]

# ── Total para anotación ──────────────────────────────────────
total = df_sankey["Conteo"].sum()

# ── Figura ────────────────────────────────────────────────────
fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(
        pad=25,
        thickness=22,
        line=dict(color="white", width=0.8),
        label=labels,
        color=node_colors,
        hovertemplate="<b>%{label}</b><br>Rupturas: <b>%{value:,}</b><extra></extra>",
    ),
    link=dict(
        source=source,
        target=target,
        value=value,
        color=link_colors,
        hovertemplate=(
            "<b>%{source.label}</b> → <b>%{target.label}</b><br>"
            "Rupturas: <b>%{value:,}</b><extra></extra>"
        ),
    ),
))

fig.update_layout(
    title=dict(
        text=(
            "<b>Causas de Rupturas</b>"
            "<br><sup>Responsable · Tipo de intervención · Diámetro afectado</sup>"
        ),
        x=0.5,
        xanchor="center",
        font=dict(size=18, color="#1a1a2e", family="Arial"),
    ),
    font=dict(family="Arial, sans-serif", size=11, color="#333333"),
    paper_bgcolor="white",
    height=560,
    width=1100,
    margin=dict(l=20, r=180, t=90, b=40),
    annotations=[dict(
        text=f"n = <b>{total:,}</b> rupturas registradas",
        x=0.5, y=-0.06,
        xref="paper", yref="paper",
        showarrow=False,
        font=dict(size=11, color="#888888", family="Arial"),
    )],
)

fig.show()

In [87]:
# ── Colores nodos usando tu lista "colores" ───────────────────
# Asigna un color de tu paleta a cada label único
color_map = {label: colores[i % len(colores)] for i, label in enumerate(labels)}
node_colors = [color_map.get(l, "#CFD8DC") for l in labels]

# ── Colores links: hereda color del nodo origen con opacidad ──
def hex_to_rgba(hex_color, alpha=0.25):
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

link_colors = [hex_to_rgba(node_colors[s]) for s in source]

# ── Total para anotación ──────────────────────────────────────
total = df_sankey["Conteo"].sum()

# ── Figura ────────────────────────────────────────────────────
fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(
        pad=25,
        thickness=22,
        line=dict(color="white", width=0.8),
        label=labels,
        color=node_colors,
        hovertemplate="<b>%{label}</b><br>Rupturas: <b>%{value:,}</b><extra></extra>",
    ),
    link=dict(
        source=source,
        target=target,
        value=value,
        color=link_colors,
        hovertemplate=(
            "<b>%{source.label}</b> → <b>%{target.label}</b><br>"
            "Rupturas: <b>%{value:,}</b><extra></extra>"
        ),
    ),
))

# ── Layout usando tu plantilla ────────────────────────────────
fig.update_layout(
    template="mi_estilo",          # ✅ Tu plantilla predefinida
    title=dict(
        text=(
            "<b>Causas de Rupturas</b>"
            "<br><sup>Responsable · Tipo de intervención · Diámetro afectado</sup>"
        ),
    ),
    height=560,
    width=1100,
    margin=dict(l=20, r=180, t=90, b=40),
    annotations=[dict(
        text=f"n = <b>{total:,}</b> rupturas registradas",
        x=0.5, y=-0.06,
        xref="paper", yref="paper",
        showarrow=False,
        font=dict(size=11, color="rgba(255,255,255,0.3)", family="Arial"),
    )],
)

fig.write_image("sankey.png", scale=2, width=1100, height=560)
fig.show()

/tmp/ipykernel_16529/3642093095.py:61: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image("sankey.png", scale=2, width=1100, height=560)


In [93]:
BATCH_SIZE = 20
CHECKPOINT = "checkpoint_maquinaria.json"

SYSTEM_PROMPT = """Eres un experto en obras civiles e infraestructura urbana.
Analiza cada observación de emergencia y determina si se usó maquinaria amarilla.

Maquinaria amarilla incluye: excavadoras, retroexcavadoras, bulldozers, 
motoniveladoras, compactadoras, grúas, volquetas, miniexcavadoras, 
bobcats, maquinaria pesada, equipo pesado, etc.

Responde SOLO con JSON exacto:
{"resultados": [{"id": 0, "categoria": "..."}, ...]}

Categorías posibles (úsalas exactamente así):
- "Uso de maquinaria amarilla"
- "No se usó maquinaria amarilla"  
- "No se pudo determinar"
"""

def clasificar_lote(textos):
    response = client.chat.completions.create(
        model="grok-3-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": json.dumps(textos, ensure_ascii=False)},
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )
    
    raw = response.choices[0].message.content  # ← captura antes de parsear
    
    if not raw or raw.strip() == "":
        print("⚠️  Respuesta vacía")
        return []

    return json.loads(raw)["resultados"]

# ── Cargar checkpoint si existe (reanuda donde quedó) ─────────
if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT) as f:
        clasificaciones = {int(k): v for k, v in json.load(f).items()}
    print(f"✅ Checkpoint cargado: {len(clasificaciones)} registros ya clasificados")
else:
    clasificaciones = {}

# ── Clasificar ────────────────────────────────────────────────
textos = df["Observaciones Emergencia"].fillna("Sin información").tolist()
total_lotes = (len(textos) + BATCH_SIZE - 1) // BATCH_SIZE

for inicio in tqdm(range(0, len(textos), BATCH_SIZE), desc="Clasificando", total=total_lotes):
    
    # Saltar lotes ya procesados
    if all(inicio + i in clasificaciones for i in range(min(BATCH_SIZE, len(textos) - inicio))):
        continue

    lote = [{"id": i, "texto": t} for i, t in enumerate(textos[inicio:inicio+BATCH_SIZE])]

    try:
        for r in clasificar_lote(lote):
            clasificaciones[inicio + r["id"]] = r["categoria"]

        # Guardar checkpoint después de cada lote exitoso
        with open(CHECKPOINT, "w") as f:
            json.dump(clasificaciones, f, ensure_ascii=False)

    except Exception as e:
        print(f"\n⚠️  Error lote {inicio}: {e}")
        #print(f"📦 Lote problemático: {json.dumps(lote[:2], ensure_ascii=False)}")  # ← primeros 2 items
        print("💾 Progreso guardado.")
        break

# ── Aplicar al DataFrame ──────────────────────────────────────
df["Maquinaria_Amarilla"] = [clasificaciones.get(i, "Pendiente") for i in range(len(textos))]

pendientes = (df["Maquinaria_Amarilla"] == "Pendiente").sum()
print(f"\n📊 Clasificados: {len(clasificaciones):,} | Pendientes: {pendientes:,}")
print(df["Maquinaria_Amarilla"].value_counts())

✅ Checkpoint cargado: 9350 registros ya clasificados


Clasificando: 100%|██████████| 468/468 [00:00<00:00, 78573.94it/s]


📊 Clasificados: 9,350 | Pendientes: 0
Maquinaria_Amarilla
No se pudo determinar            4987
Uso de maquinaria amarilla       2561
No se usó maquinaria amarilla    1802
Name: count, dtype: int64


In [94]:
df_sankey2 = (
    df.groupby(["TIPO_LIMPIO", "Maquinaria_Amarilla"])
      .size()
      .reset_index(name="Conteo")
)

labels = list(pd.concat([
    df_sankey2["TIPO_LIMPIO"],
    df_sankey2["Maquinaria_Amarilla"]
]).unique())

label_dict = {label: i for i, label in enumerate(labels)}

## Nivel 1: "TIPO_LIMPIO" → Maquinaria_Amarilla
source_1 = df_sankey2["TIPO_LIMPIO"].map(label_dict)
target_1 = df_sankey2["Maquinaria_Amarilla"].map(label_dict)
value_1  = df_sankey2["Conteo"]



In [ ]:
# ── Paleta de colores ─────────────────────────────────────────
COLORES_NODO = {
    # Nivel 1 - Responsable
    "Excavación":                               "#481066",
    "Instalación de postes y cerramientos":     "#ea1e76",
    "Alcantarillado y acueducto":               "#4a7628",
    "Pavimentación":                            "#cfde00",
    # Nivel 2 - Tipo
    "No se pudo determinar":                    "#00b4e3",
    "Uso de maquinaria amarilla":               "#05205a",
    "No se usó maquinaria amarilla":            "#DA16FF",
}
COLOR_DEFAULT = "#CFD8DC"

# ── Colores nodos ─────────────────────────────────────────────
node_colors = [COLORES_NODO.get(l, COLOR_DEFAULT) for l in labels]

# ── Colores links: hereda color del nodo origen con opacidad ──
def hex_to_rgba(hex_color, alpha=0.5):
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    return f"rgba({r},{g},{b},{alpha})"

link_colors = [hex_to_rgba(node_colors[s]) for s in source_1]

# ── Total para anotación ──────────────────────────────────────
total = df_sankey2["Conteo"].sum()

# ── Figura ────────────────────────────────────────────────────
fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(
        pad=25,
        thickness=22,
        line=dict(color="white", width=0.8),
        label=labels,
        color=node_colors,
        hovertemplate="<b>%{label}</b><br>Rupturas: <b>%{value:,}</b><extra></extra>",
    ),
    link=dict(
        source=source_1,
        target=target_1,
        value=value_1,
        color=link_colors,
        hovertemplate=(
            "<b>%{source.label}</b> → <b>%{target.label}</b><br>"
            "Rupturas: <b>%{value:,}</b><extra></extra>"
        ),
    ),
))

# ── Layout usando tu plantilla ────────────────────────────────
fig.update_layout(
    template="mi_estilo",          # ✅ Tu plantilla predefinida
    title=dict(
        text=(
            "<b>Causas de Rupturas</b>"
            "<br><sup> Tipo de intervención · Diámetro afectado</sup>"
        ),
    ),
    height=560,
    width=1100,
    margin=dict(l=20, r=180, t=90, b=40),
    annotations=[dict(
        text=f"n = <b>{total:,}</b> rupturas registradas",
        x=0.5, y=-0.06,
        xref="paper", yref="paper",
        showarrow=False,
        font=dict(size=11, color="rgba(255,255,255,0.3)", family="Arial"),
    )],
)

fig.write_image("sankey2.png", scale=2, width=1100, height=560)
fig.show()

/tmp/ipykernel_16529/318232776.py:73: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image("sankey2.png", scale=2, width=1100, height=560)


In [6]:
df["Tipo de contratista"] = (df["Tipo de contratista"].fillna("NO IDENTIFICADO").replace("", "NO IDENTIFICADO").str.strip().replace("", "NO IDENTIFICADO")) 

df_tipo_contra = (df.groupby(["Tipo de contratista"]).size().reset_index(name="Conteo").sort_values("Conteo", ascending=False))

fig = px.treemap(
    df_tipo_contra,
    path=[px.Constant("Total"), "Tipo de contratista"],
    values="Conteo",
    color="Conteo",
    color_continuous_scale="Blues",
    title="<b>Rupturas por Tipo de Contratista</b>",
)

fig.update_traces(
    texttemplate="<b>%{label}</b><br>%{value:,}",
    hovertemplate="<b>%{label}</b><br>Rupturas: %{value:,}<br>%{percentRoot:.1%} del total<extra></extra>",
)

fig.update_layout(
    margin=dict(t=60, l=15, r=15, b=15),
    paper_bgcolor="white",
    title_x=0.5,
    font=dict(family="Arial", size=12),
    coloraxis_colorbar=dict(title="Rupturas"),
)

fig.show()

In [7]:
df_vacios = df[df["Tipo de contratista"] == "-"].groupby(["Empresa que ocasiono daño"]).size().reset_index(name="Conteo").sort_values("Conteo", ascending=False)

fig = px.treemap(
    df_vacios,
    path=[px.Constant("Total"), "Empresa que ocasiono daño"],
    values="Conteo",
    color="Conteo",
    color_continuous_scale="Blues",
    title="<b>Rupturas por Empresa que ocasiono daño</b>",
)

fig.update_traces(
    texttemplate="<b>%{label}</b><br>%{value:,}",
    hovertemplate="<b>%{label}</b><br>Rupturas: %{value:,}<br>%{percentRoot:.1%} del total<extra></extra>",
)

fig.update_layout(
    margin=dict(t=60, l=15, r=15, b=15),
    paper_bgcolor="white",
    title_x=0.5,
    font=dict(family="Arial", size=12),
    coloraxis_colorbar=dict(title="Rupturas"),
)

fig.show()

In [8]:
df_vacios2 = df[df["Tipo de contratista"] == "NO IDENTIFICADO"].groupby(["Empresa que ocasiono daño"]).size().reset_index(name="Conteo").sort_values("Conteo", ascending=False)

fig = px.treemap(
    df_vacios2,
    path=[px.Constant("Total"), "Empresa que ocasiono daño"],
    values="Conteo",
    color="Conteo",
    color_continuous_scale="Blues",
    title="<b>Rupturas Empresa que ocasiono daño</b>",
)

fig.update_traces(
    texttemplate="<b>%{label}</b><br>%{value:,}",
    hovertemplate="<b>%{label}</b><br>Rupturas: %{value:,}<br>%{percentRoot:.1%} del total<extra></extra>",
)

fig.update_layout(
    margin=dict(t=60, l=15, r=15, b=15),
    paper_bgcolor="white",
    title_x=0.5,
    font=dict(family="Arial", size=12),
    coloraxis_colorbar=dict(title="Rupturas"),
)

fig.show()


In [9]:
df_tipo_expe = (df.groupby(["Tipo de contratista", "Empresa que ocasiono daño"]).size().reset_index(name="Conteo").sort_values("Conteo", ascending=False))

fig = px.treemap(
    df_tipo_expe,
    path=[px.Constant("Total"), "Tipo de contratista", "Empresa que ocasiono daño"], 
    values="Conteo",
    color="Conteo",
    color_continuous_scale="Blues",
    title="<b>Rupturas por Tipo de contratista y Empresa que ocasiono daño</b>",
)

fig.update_traces(
    texttemplate="<b>%{label}</b><br>%{value:,}",
    hovertemplate="<b>%{label}</b><br>Rupturas: %{value:,}<br>%{percentRoot:.1%} del total<extra></extra>",
)

fig.update_layout(
    margin=dict(t=60, l=15, r=15, b=15),
    title_x=0.5,
    font=dict(family="Arial", size=12),
    coloraxis_showscale=False,
)
fig.write_image("plot7.png", scale=2, width=1200, height=600)
fig.write_html("grafico.html")
fig.show()

/tmp/ipykernel_1891/1388749715.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image("plot7.png", scale=2, width=1200, height=600)


In [110]:
df_tipo_contra = (df.groupby(["Tipo de contratista"]).size().reset_index(name="Conteo").sort_values("Conteo", ascending=False))

fig = px.pie(
    df_tipo_contra,
    names="Tipo de contratista",
    values="Conteo",
    color_discrete_sequence=colores,
    title="Rupturas por tipo de contratista",
    hole=0.4,          # ✅ Tamaño del hueco: 0 = pie, 1 = sin relleno. Prueba entre 0.3 y 0.5
)

fig.update_traces(
    textposition="inside",
    textinfo="percent",   # ✅ Opciones: "percent", "label", "value", "percent+label"
    textfont=dict(color="white", size=11, family="Arial"),
)

fig.update_layout(
    showlegend=True,
    height=600,
    width=700,
)

fig.write_image("plot6.png", scale=2, width=800, height=600)
fig.show()

/tmp/ipykernel_16529/1807787011.py:24: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image("plot6.png", scale=2, width=800, height=600)
